# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import json, os, warnings
import numpy as np
import pandas as pd
import duckdb
import sklearn
from google.colab import userdata

SEED = 42                     # every random step below uses this seed
FEATURE_MONTH = "2026-03"     # cluster on this month (same as Weeks 3-4)
OUTCOME_MONTH = "2026-04"     # judge persistence on this LATER month; never used as a feature
                              # (June 2026 is the sealed final month: not touched here)
VOLUME_FLOOR = 10             # same floor as the Week-4 gate
VISIBILITY_CUTOFF = 0.10      # same cutoff as the Week-4 gate
EXPECTED_VALID, EXPECTED_FLAGGED = 90237, 220   # Week-4 receipts this notebook must reproduce
REL = "hf://datasets/FlyRank/internship-warehouse"
np.random.seed(SEED)

con = duckdb.connect()
token = userdata.get("HF_TOKEN")
con.sql(f"""
    CREATE OR REPLACE SECRET hf_token (
        TYPE huggingface,
        TOKEN '{token}'
    )
""")
assert len(con.sql("SELECT * FROM duckdb_secrets()").df()) > 0, "HF secret not registered"

def load_month(month):
    """One row per (client, content) for one month: same aggregation as Weeks 3-4."""
    path = f"{REL}/fact_content_daily_performance/month={month}/*.parquet"
    q = f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks)::DOUBLE / NULLIF(SUM(gsc_impressions), 0)             AS ctr,
        SUM(ga4_engaged_sessions)::DOUBLE / NULLIF(SUM(ga4_sessions), 0)      AS engagement_rate,
        SUM(ga4_total_engagement_sec)::DOUBLE / NULLIF(SUM(ga4_sessions), 0)  AS session_depth_sec,
        SUM(sessions_ai)::DOUBLE / NULLIF(
            SUM(sessions_organic + sessions_direct + sessions_referral
                + sessions_social + sessions_paid + sessions_ai), 0)          AS ai_traffic_pct,
        AVG((gsc_data_available IS TRUE)::INT)                                 AS gsc_availability_rate,
        SUM(ga4_sessions)                                                      AS sessions_month,
        SUM(ga4_engaged_sessions)                                              AS engaged_sessions_month
    FROM read_parquet('{path}')
    GROUP BY client_hash_id, content_hash_id
    """
    return con.sql(q).df()

march = load_month(FEATURE_MONTH)
valid = march.dropna(subset=["engagement_rate", "gsc_availability_rate"]).copy()
valid["passes_gate"] = ((valid["sessions_month"] >= VOLUME_FLOOR)
                        & (valid["gsc_availability_rate"] <= VISIBILITY_CUTOFF))

# Receipts: this must be the SAME data and the SAME baseline as Week 4, or the comparison means nothing.
assert len(valid) == EXPECTED_VALID, f"valid rows {len(valid)} != Week-4 {EXPECTED_VALID}"
assert int(valid["passes_gate"].sum()) == EXPECTED_FLAGGED, "baseline flagged count differs from Week 4"
print(f"Week-4 receipts reproduced: {len(valid)} valid pages, {int(valid['passes_gate'].sum())} flagged by the baseline.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Week-4 receipts reproduced: 90237 valid pages, 220 flagged by the baseline.


In [2]:
# The shared population: pages whose rates rest on >= VOLUME_FLOOR sessions.
# Below the floor, rates are mass points (Week 4: 1-session pages are 100% extreme rates), so
# NEITHER the baseline nor the model gets to make claims about them. The floor is a data rule
# both arms share, not a modelling choice.
pop = valid[valid["sessions_month"] >= VOLUME_FLOOR].copy()
assert pop.loc[pop["passes_gate"]].shape[0] == EXPECTED_FLAGGED, "gate must sit fully inside the floor population"

# Model inputs = everything the baseline saw (engagement_rate, sessions, gsc_availability_rate)
# plus session depth from the Week-3 contract. A model that sees LESS than the baseline can't
# fairly lose to it. log1p tames the heavy tails so K-Means isn't ruled by a few huge pages.
X_raw = pd.DataFrame({
    "engagement_rate":        pop["engagement_rate"],
    "gsc_availability_rate":  pop["gsc_availability_rate"],
    "log_sessions":           np.log1p(pop["sessions_month"]),
    "log_session_depth_sec":  np.log1p(pop["session_depth_sec"]),
})
FEATURES = list(X_raw.columns)
keep = X_raw.notna().all(axis=1)
print(f"Rows dropped for missing model inputs: {int((~keep).sum())}")
pop, X_raw = pop.loc[keep].copy(), X_raw.loc[keep].copy()
assert int(pop["passes_gate"].sum()) == EXPECTED_FLAGGED, "dropping NaN rows removed baseline-flagged pages"
pop = pop.reset_index(drop=True); X_raw = X_raw.reset_index(drop=True)

# Privacy: never print raw client hashes. Relabel clients C1, C2, ... by size.
order = pop["client_hash_id"].value_counts().index
pop["client"] = pop["client_hash_id"].map({c: f"C{i+1}" for i, c in enumerate(order)})

print(f"Population: {len(pop)} pages, {pop['client_hash_id'].nunique()} clients, "
      f"{int(pop['passes_gate'].sum())} flagged by the baseline ({pop['passes_gate'].mean():.2%}).")
print("Model features:", FEATURES)

# Why not the full Week-3 five-feature frame? ctr / ai_traffic_pct are undefined for pages with no
# impressions / no traffic-mix data, and low-visibility pages are exactly the ones most likely to lack them.
W3_FEATURES = ["ctr", "engagement_rate", "session_depth_sec", "ai_traffic_pct", "gsc_availability_rate"]
cc = pop.dropna(subset=W3_FEATURES)
print(f"\nComplete cases on the 5-feature Week-3 frame: {len(cc)} of {len(pop)} pages; "
      f"baseline-flagged pages that would survive: {int(cc['passes_gate'].sum())} of {EXPECTED_FLAGGED}.")

Rows dropped for missing model inputs: 0
Population: 21207 pages, 32 clients, 220 flagged by the baseline (1.04%).
Model features: ['engagement_rate', 'gsc_availability_rate', 'log_sessions', 'log_session_depth_sec']

Complete cases on the 5-feature Week-3 frame: 20616 of 21207 pages; baseline-flagged pages that would survive: 34 of 220.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

N_FOLDS = min(5, pop["client_hash_id"].nunique())
folds = list(GroupKFold(n_splits=N_FOLDS).split(X_raw, groups=pop["client_hash_id"]))

# Leakage guard: a client must never sit on both sides of a split.
for tr, te in folds:
    assert set(pop["client_hash_id"].iloc[tr]).isdisjoint(pop["client_hash_id"].iloc[te])

fold_tbl = pd.DataFrame([{
    "fold": i,
    "n_pages": len(te),
    "n_clients": pop["client_hash_id"].iloc[te].nunique(),
    "n_baseline_flagged": int(pop["passes_gate"].iloc[te].sum()),
    "largest_client_share": pop["client"].iloc[te].value_counts(normalize=True).iloc[0],
} for i, (tr, te) in enumerate(folds)]).round(3)
print("Held-out client folds (every page is held out exactly once):")
print(fold_tbl.to_string(index=False))

Held-out client folds (every page is held out exactly once):
 fold  n_pages  n_clients  n_baseline_flagged  largest_client_share
    0     9337          1                   6                 1.000
    1     3928          1                   2                 1.000
    2     2648         10                 125                 0.791
    3     2647         10                  33                 0.521
    4     2647         10                  54                 0.486


In [4]:
def heldout_scores(k):
    """Fit scaler + K-Means on TRAIN clients only; score on clients the model has never seen."""
    rows = []
    for tr, te in folds:
        sc = StandardScaler().fit(X_raw.iloc[tr])
        Xtr, Xte = sc.transform(X_raw.iloc[tr]), sc.transform(X_raw.iloc[te])
        km = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(Xtr)
        lab = km.predict(Xte)
        own = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(Xte).labels_   # structure the held-out clients have on their own
        sil = (silhouette_score(Xte, lab, sample_size=min(len(Xte), 5000), random_state=SEED)
               if len(set(lab)) > 1 else np.nan)
        rows.append({"k": k, "silhouette": sil,
                     "ari_transfer": adjusted_rand_score(own, lab),          # do train-fit clusters match held-out structure?
                     "smallest_cluster_share": np.bincount(lab, minlength=k).min() / len(lab)})
    return pd.DataFrame(rows)

k_tbl = (pd.concat([heldout_scores(k) for k in range(2, 9)])
           .groupby("k").agg(silhouette_mean=("silhouette", "mean"), silhouette_sd=("silhouette", "std"),
                             ari_transfer_mean=("ari_transfer", "mean"),
                             smallest_cluster_share=("smallest_cluster_share", "min"))
           .round(3))
# Silhouette often favours k=2 when a feature is bimodal (gsc_availability_rate is). Treat it as ONE vote:
# weigh it with ari_transfer, smallest_cluster_share, and whether the profiles below are nameable.
print("Choice of k, judged on held-out clients (mean over folds):")
print(k_tbl)

K_OVERRIDE = None    # set an int here after reading the table if a smaller/larger k tells the story better
K = int(K_OVERRIDE or k_tbl["silhouette_mean"].idxmax())
print(f"\nUsing k = {K}")

Choice of k, judged on held-out clients (mean over folds):
   silhouette_mean  silhouette_sd  ari_transfer_mean  smallest_cluster_share
k                                                                           
2            0.433          0.069              0.499                   0.175
3            0.331          0.057              0.623                   0.061
4            0.398          0.056              0.670                   0.005
5            0.382          0.053              0.572                   0.005
6            0.380          0.068              0.648                   0.002
7            0.368          0.072              0.568                   0.002
8            0.356          0.052              0.642                   0.002

Using k = 2


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist

# Reference model on ALL clients: only used to define/profile the clusters and give fold-clusters common ids.
ref_sc = StandardScaler().fit(X_raw)
ref_km = KMeans(n_clusters=K, n_init=10, random_state=SEED).fit(ref_sc.transform(X_raw))
pop["cluster_ref"] = ref_km.labels_

# Out-of-fold assignment: each page is labelled by a model that never saw its client.
pop["cluster_oof"] = -1
for tr, te in folds:
    sc = StandardScaler().fit(X_raw.iloc[tr])
    km = KMeans(n_clusters=K, n_init=10, random_state=SEED).fit(sc.transform(X_raw.iloc[tr]))
    # fold cluster ids are arbitrary; match them to the reference clusters by centroid distance (Hungarian)
    centers = pd.DataFrame(sc.inverse_transform(km.cluster_centers_), columns=FEATURES)
    cost = cdist(ref_sc.transform(centers), ref_km.cluster_centers_)
    r, c = linear_sum_assignment(cost)
    mapping = dict(zip(r, c))
    pop.loc[te, "cluster_oof"] = [mapping[l] for l in km.predict(sc.transform(X_raw.iloc[te]))]
assert (pop["cluster_oof"] >= 0).all()
print(f"Agreement of out-of-fold clusters with the all-client clusters (ARI): "
      f"{adjusted_rand_score(pop['cluster_ref'], pop['cluster_oof']):.3f}  (optimistic: the reference model saw these pages)")

# Cluster profiles in real units. READ THIS TABLE BEFORE NAMING ANYTHING.
profile = pop.groupby("cluster_ref").agg(
    n_pages=("client", "size"), n_clients=("client", "nunique"),
    engagement_rate=("engagement_rate", "median"), gsc_availability=("gsc_availability_rate", "median"),
    sessions=("sessions_month", "median"), session_depth_sec=("session_depth_sec", "median"),
    baseline_flagged=("passes_gate", "sum")).round(3)
print("\nCluster profiles (medians):")
print(profile)
print("\nWhere the baseline's flagged pages land (rows = out-of-fold cluster):")
print(pd.crosstab(pop["cluster_oof"], pop["passes_gate"].map({True: "flagged", False: "not flagged"})))

Agreement of out-of-fold clusters with the all-client clusters (ARI): 0.932  (optimistic: the reference model saw these pages)

Cluster profiles (medians):
             n_pages  n_clients  engagement_rate  gsc_availability  sessions  \
cluster_ref                                                                    
0              16088         31            0.000               1.0      28.0   
1               5119         30            0.074               1.0      24.0   

             session_depth_sec  baseline_flagged  
cluster_ref                                       
0                        0.423               106  
1                       10.061               114  

Where the baseline's flagged pages land (rows = out-of-fold cluster):
passes_gate  flagged  not flagged
cluster_oof                      
0                106        15792
1                114         5195


In [6]:
# Which cluster is the "hidden gem" cluster? YOU decide after reading the profile table above.
# Default rule (independent of the baseline, taken from the hypothesis itself): among clusters with
# above-population-median engagement, the one with the lowest gsc availability.
GEM_CLUSTER_OVERRIDE = None     # e.g. 2 -- set after inspecting the profiles
pop_median_eng = pop["engagement_rate"].median()
cand = profile[profile["engagement_rate"] >= pop_median_eng]
GEM_CLUSTER = (GEM_CLUSTER_OVERRIDE if GEM_CLUSTER_OVERRIDE is not None
               else (int(cand["gsc_availability"].idxmin()) if len(cand) else None))
print("Gem cluster:", GEM_CLUSTER if GEM_CLUSTER is not None else "none: no cluster matches the hypothesis (a valid result)")
pop["model_gem"] = (pop["cluster_oof"] == GEM_CLUSTER) if GEM_CLUSTER is not None else False

Gem cluster: 0


In [7]:
# Outcome window: a LATER month than the one used to build clusters. No overlap, no leakage.
apr = load_month(OUTCOME_MONTH)[["client_hash_id", "content_hash_id", "sessions_month", "engaged_sessions_month"]]
apr = apr.rename(columns={"sessions_month": "sessions_apr", "engaged_sessions_month": "engaged_apr"})
ev = pop.merge(apr, on=["client_hash_id", "content_hash_id"], how="left")
assert len(ev) == len(pop), "merge changed the row count"
ev[["sessions_apr", "engaged_apr"]] = ev[["sessions_apr", "engaged_apr"]].fillna(0)

def rank_by_score(df):
    # the Week-4 score, held FIXED for both arms: engagement_rate, ties by volume.
    # So the only thing that differs between the arms is WHO IS ELIGIBLE (hand gate vs learned cluster).
    return df.sort_values(["engagement_rate", "sessions_month"], ascending=[False, False])

def set_metrics(df, name):
    s_m, e_m = df["sessions_month"].sum(), df["engaged_sessions_month"].sum()
    s_a, e_a = df["sessions_apr"].sum(), df["engaged_apr"].sum()
    return {"set": name, "n_pages": len(df), "n_clients": df["client"].nunique(),
            "largest_client_share": df["client"].value_counts(normalize=True).iloc[0],
            "mar_engagement (selection window)": e_m / s_m,
            "apr_still_>=10_sessions": (df["sessions_apr"] >= VOLUME_FLOOR).mean(),
            "apr_engagement (outcome)": e_a / s_a if s_a > 0 else np.nan}

base_all = ev[ev["passes_gate"]]
model_all = ev[ev["model_gem"]]
rows = [set_metrics(ev, "All floor pages (base rate)"),
        set_metrics(base_all, "Baseline: all flagged")]
for K_TOP in (50, 20):
    rows.append(set_metrics(rank_by_score(base_all).head(K_TOP), f"Baseline: top-{K_TOP}"))
if len(model_all):
    rows.append(set_metrics(model_all, "Model gem cluster: all"))
    for K_TOP in (50, 20):
        rows.append(set_metrics(rank_by_score(model_all).head(K_TOP), f"Model gem cluster: top-{K_TOP}"))
# Where the two arms DISAGREE is the most informative part: if the sets are nested, top-K rows can be identical.
only_model = ev[ev["model_gem"] & ~ev["passes_gate"]]
only_base = ev[ev["passes_gate"] & ~ev["model_gem"]]
if len(only_model): rows.append(set_metrics(only_model, "Model gem cluster ONLY (not flagged)"))
if len(only_base): rows.append(set_metrics(only_base, "Baseline flagged ONLY (not in cluster)"))
compare = pd.DataFrame(rows).set_index("set").round(3)
print(f"Model vs baseline: same {len(ev)} pages, same held-out clients, same score, outcome = {OUTCOME_MONTH}")
print(compare.to_string())

overlap = (base_all.index.intersection(model_all.index))
print(f"\nOverlap: {len(overlap)} pages are in BOTH the baseline set and the model gem cluster "
      f"(baseline {len(base_all)}, model {len(model_all)}).")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Model vs baseline: same 21207 pages, same held-out clients, same score, outcome = 2026-04
                                        n_pages  n_clients  largest_client_share  mar_engagement (selection window)  apr_still_>=10_sessions  apr_engagement (outcome)
set                                                                                                                                                                   
All floor pages (base rate)               21207         32                 0.440                              0.022                    0.579                     0.038
Baseline: all flagged                       220         23                 0.373                              0.058                    0.464                     0.063
Baseline: top-50                             50          8                 0.600                              0.166                    0.580                     0.087
Baseline: top-20                             20          6                 

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Check 1: is "low visibility" a property of the PAGE, or of when the client's Search Console connected?
dim = con.sql(f"SELECT * FROM read_parquet('{REL}/dim_clients.parquet')").df()
assert "client_hash_id" in dim.columns, f"unexpected dim_clients columns: {list(dim.columns)}"
month_start = pd.Timestamp(f"{FEATURE_MONTH}-01")
late = dim.assign(gsc_late=pd.to_datetime(dim["gsc_data_start"]) > month_start)[["client_hash_id", "gsc_late"]]
ev = ev.drop(columns=["gsc_late"], errors="ignore").merge(late, on="client_hash_id", how="left")
ev["gsc_late"] = ev["gsc_late"].fillna(False).astype(bool)

def late_share(df): return df["gsc_late"].mean() if len(df) else np.nan
print("Share of pages from clients whose Search Console data STARTS AFTER the March window began:")
print(f"  all floor pages:        {late_share(ev):.1%}")
print(f"  baseline flagged ({EXPECTED_FLAGGED}): {late_share(ev[ev['passes_gate']]):.1%}")
print(f"  model gem cluster:      {late_share(ev[ev['model_gem']]):.1%}")

# Check 2: three concrete wrong cases: top-20 picks from each arm whose engagement did not persist.
def worst_cases(df, n=3):
    top = rank_by_score(df).head(20).copy()
    top["apr_rate"] = np.where(top["sessions_apr"] > 0, top["engaged_apr"] / top["sessions_apr"].where(top["sessions_apr"] > 0), np.nan)
    top["drop"] = top["engagement_rate"] - top["apr_rate"].fillna(0)
    return (top.sort_values("drop", ascending=False).head(n)
               [["client", "sessions_month", "engagement_rate", "sessions_apr", "apr_rate"]].round(3))
print("\nBaseline top-20: three biggest misses")
print(worst_cases(ev[ev["passes_gate"]]).to_string(index=False))
if ev["model_gem"].any():
    print("\nModel gem cluster top-20: three biggest misses")
    print(worst_cases(ev[ev["model_gem"]]).to_string(index=False))

Share of pages from clients whose Search Console data STARTS AFTER the March window began:
  all floor pages:        0.1%
  baseline flagged (220): 0.5%
  model gem cluster:      0.1%

Baseline top-20: three biggest misses
client  sessions_month  engagement_rate  sessions_apr  apr_rate
   C15            13.0            0.308           9.0       0.0
   C15            10.0            0.300           6.0       0.0
   C11            21.0            0.286           1.0       0.0

Model gem cluster top-20: three biggest misses
client  sessions_month  engagement_rate  sessions_apr  apr_rate
    C1            11.0            0.091           1.0       0.0
    C1            12.0            0.083           6.0       0.0
    C5            12.0            0.083           7.0       0.0


In [9]:
# Check 3: what does the clustering lean on? A depth-3 tree that imitates the clusters gives readable rules.
# Caution: high fidelity here is EXPECTED (clusters are functions of these features). It says the rules are
# a faithful summary, not that the clusters are valid.
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import cross_val_predict
from sklearn.inspection import permutation_importance

surrogate = DecisionTreeClassifier(max_depth=3, random_state=SEED)
pred = cross_val_predict(surrogate, X_raw, pop["cluster_ref"], groups=pop["client_hash_id"],
                         cv=GroupKFold(n_splits=N_FOLDS))
print(f"Surrogate-tree fidelity on held-out clients: {(pred == pop['cluster_ref']).mean():.3f}")
surrogate.fit(X_raw, pop["cluster_ref"])
print(export_text(surrogate, feature_names=FEATURES))

tr, te = folds[0]
tree0 = DecisionTreeClassifier(max_depth=3, random_state=SEED).fit(X_raw.iloc[tr], pop["cluster_ref"].iloc[tr])
pi = permutation_importance(tree0, X_raw.iloc[te], pop["cluster_ref"].iloc[te], n_repeats=20, random_state=SEED)
print("Permutation importance (held-out clients, accuracy drop when a feature is shuffled):")
print(pd.Series(pi.importances_mean, index=FEATURES).sort_values(ascending=False).round(3))

Surrogate-tree fidelity on held-out clients: 0.979
|--- log_session_depth_sec <= 1.59
|   |--- engagement_rate <= 0.06
|   |   |--- engagement_rate <= 0.05
|   |   |   |--- class: 0
|   |   |--- engagement_rate >  0.05
|   |   |   |--- class: 0
|   |--- engagement_rate >  0.06
|   |   |--- log_session_depth_sec <= 1.05
|   |   |   |--- class: 0
|   |   |--- log_session_depth_sec >  1.05
|   |   |   |--- class: 1
|--- log_session_depth_sec >  1.59
|   |--- engagement_rate <= 0.03
|   |   |--- log_session_depth_sec <= 2.07
|   |   |   |--- class: 0
|   |   |--- log_session_depth_sec >  2.07
|   |   |   |--- class: 1
|   |--- engagement_rate >  0.03
|   |   |--- engagement_rate <= 0.04
|   |   |   |--- class: 1
|   |   |--- engagement_rate >  0.04
|   |   |   |--- class: 1

Permutation importance (held-out clients, accuracy drop when a feature is shuffled):
log_session_depth_sec    0.193
engagement_rate          0.047
gsc_availability_rate    0.000
log_sessions             0.000
dtype: fl

In [10]:
# Receipts: commit this file (work/outputs/*.json is meant to be committed, unlike CSVs).
os.makedirs("work/outputs", exist_ok=True)
receipt = {
    "seed": SEED, "k": K, "gem_cluster": None if GEM_CLUSTER is None else int(GEM_CLUSTER),
    "feature_month": FEATURE_MONTH, "outcome_month": OUTCOME_MONTH,
    "volume_floor": VOLUME_FLOOR, "visibility_cutoff": VISIBILITY_CUTOFF,
    "n_pages": int(len(pop)), "features": FEATURES,
    "versions": {"sklearn": sklearn.__version__, "pandas": pd.__version__, "numpy": np.__version__},
    "k_selection": json.loads(k_tbl.reset_index().to_json(orient="records")),
    "comparison": json.loads(compare.reset_index().to_json(orient="records")),
}
with open("work/outputs/w05_model_metrics.json", "w") as f:
    json.dump(receipt, f, indent=2)

# Self-check the parts a reviewer can verify mechanically:
assert any(s.startswith("Baseline") for s in compare.index), "baseline must appear in the same table as the model"
assert (pop["cluster_oof"] >= 0).all() and pop["client_hash_id"].nunique() >= N_FOLDS
print("Baseline and model share one table, one population, one held-out-client split. Saved work/outputs/w05_model_metrics.json")

Baseline and model share one table, one population, one held-out-client split. Saved work/outputs/w05_model_metrics.json


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.